# Bozeman Dataset 

### Cleaning & Initial Preprocessing of Bozeman dataset (Joe Tambeana)

This notebook will be for cleaning and pre-processing at the next step for the Bozeman and kitchener datasets

The goal is
- to understand the datatypes that each dataset holds
- clean and drop out columns or rows that are not relevant for analysis
- identify where they can be linked for analysis
- get all cleaned ready for preprocessing

### An overview of the dataset

- information about break date, installation date, pipe size,type of leak are critical for analysis 

In [4]:
# importing the datasets

import pandas as pd
bozemanDataFrame = pd.read_csv("../data/raw/Bozeman/Bozeman_watermainbreaks.csv")


# print the first 10 rows of each dataset
print("=== Bozeman Water Main Breaks Dataset ===")
print(bozemanDataFrame.head(10).to_string())


=== Bozeman Water Main Breaks Dataset ===
               X             Y  OBJECTID_1              Break_Date                              Address  Pipe_Size                     Type_of_Leak     Cost     WOID   FACID  PIPE_INSTALL_DA Detected
0  496675.559596  5.058910e+06           1  2013/01/28 00:00:00+00                            431 N 4TH        6.0                        Corrosion  3273.33  86318.0   334.0           1948.0       No
1  496631.156500  5.059796e+06           2  2012/08/11 00:00:00+00                   1126 N 7TH (KMART)        8.0               Longitudinal Break   286.10  80400.0   204.0           1974.0       No
2  496352.228196  5.060142e+06           3  2002/02/15 00:00:00+00            5 Baxter Lane Holiday Inn        4.0             Pipe Failure at Bend   770.57      NaN   281.0           2001.0       No
3  496383.946396  5.058829e+06           4  1997/07/07 00:00:00+00            Seventh Av N/Villard St W        6.0  Rock in hyd nozzel - hammer N/C  2301.76  

### Overview of the 3 datasets

In [6]:
# quick overview of the dataset

print("\nBozeman Water Main Breaks info:")
bozemanDataFrame.info()
print("\nKitchener Water Main Breaks info:")



Bozeman Water Main Breaks info:
<class 'pandas.core.frame.DataFrame'>
RangeIndex: 158 entries, 0 to 157
Data columns (total 12 columns):
 #   Column           Non-Null Count  Dtype  
---  ------           --------------  -----  
 0   X                158 non-null    float64
 1   Y                158 non-null    float64
 2   OBJECTID_1       158 non-null    int64  
 3   Break_Date       158 non-null    object 
 4   Address          153 non-null    object 
 5   Pipe_Size        157 non-null    float64
 6   Type_of_Leak     158 non-null    object 
 7   Cost             156 non-null    float64
 8   WOID             122 non-null    float64
 9   FACID            157 non-null    float64
 10  PIPE_INSTALL_DA  155 non-null    float64
 11  Detected         158 non-null    object 
dtypes: float64(7), int64(1), object(4)
memory usage: 14.9+ KB

Kitchener Water Main Breaks info:


### Check statistics for missing values

In [8]:

# print the count and percentage of missing values for each column

# Define constants
MISSING_COUNT = 'Missing Count'
MISSING_PERCENT = 'Missing %'
NON_NULL_COUNT = 'Non-Null Count'

# using the dataframe already loaded for Bozeman dataset
print("\n=== Bozeman Missing Values Analysis ===")
dfBozeman = bozemanDataFrame  
missing = pd.DataFrame({
    MISSING_COUNT: dfBozeman.isna().sum(),
    MISSING_PERCENT: (dfBozeman.isna().sum() / len(dfBozeman) * 100).round(2),
    NON_NULL_COUNT: dfBozeman.notna().sum()
})
print(missing)



=== Bozeman Missing Values Analysis ===
                 Missing Count  Missing %  Non-Null Count
X                            0       0.00             158
Y                            0       0.00             158
OBJECTID_1                   0       0.00             158
Break_Date                   0       0.00             158
Address                      5       3.16             153
Pipe_Size                    1       0.63             157
Type_of_Leak                 0       0.00             158
Cost                         2       1.27             156
WOID                        36      22.78             122
FACID                        1       0.63             157
PIPE_INSTALL_DA              3       1.90             155
Detected                     0       0.00             158


### Check for negative values

In [10]:
# Check for negative values in numeric columns of Bozeman dataset
bozemannegatives = total_negatives = (dfBozeman.select_dtypes(include='number') < 0).sum().sum()
bozemannegatives

print(f"Total negative values in Bozeman dataset: {bozemannegatives}")


Total negative values in Bozeman dataset: 0


### Deplicates check for the datasets

In [12]:
#duplicate analysis 

# Assuming your dataframe is called df
print("=== Bozeman Duplicate Check ===")

# Total number of duplicate rows
total_duplicates = dfBozeman.duplicated().sum()
print(f"Total duplicate rows: {total_duplicates}")

# Percentage of duplicates
if len(dfBozeman) > 0:
    duplicate_pct = (total_duplicates / len(dfBozeman) * 100).round(2)
    print(f"Duplicate percentage: {duplicate_pct}%")


=== Bozeman Duplicate Check ===
Total duplicate rows: 0
Duplicate percentage: 0.0%


### Data cleaning

In [14]:
# # # ============================== Bozeman Dataset Cleaning =====================================
def clean_data(df):
    # Drop rows with missing data in key columns
    df = df.dropna(subset=['FACID', 'WOID', 'Cost', 'PIPE_INSTALL_DA', 
                           'Pipe_Size', 'Address'])

    # Convert 'Break_Date' to datetime and remove timezone
    df['Break_Date'] = pd.to_datetime(df['Break_Date'], errors='coerce')
    df['Break_Date'] = df['Break_Date'].dt.tz_localize(None)

    # Create Install_Year from PIPE_INSTALL_DA and drop original column
    df['Install_Year'] = df['PIPE_INSTALL_DA'].astype(int)
    df = df.drop(columns=['PIPE_INSTALL_DA'])

    # Drop low-relevance columns
    df = df.drop(columns=['Cost', 'WOID', 'OBJECTID_1'])

    # ====================== Rename Columns ======================
    df = df.rename(columns={
        'FACID': 'Pipe_ID',
        'Pipe_Size': 'Pipe_Size_inch',
        'Type_of_Leak': 'Leak_Type'
    })

    # ====================== Create Requested Features ======================
    # Break year
    df['Break_Year'] = df['Break_Date'].dt.year

    # Age at Break
    df['Age_at_Break'] = df['Break_Year'] - df['Install_Year']

    # Age Squared (for non-linear effects)
    df['Age_Squared'] = df['Age_at_Break'] ** 2

    # Standardized Leak Type
    leak_mapping = {
        'Full Circle Break': 'Circumferential',
        'Full Circle': 'Circumferential',
        'Longitudinal Break': 'Longitudinal',
        'Leaking Joint': 'Joint',
        'Blow-out': 'Blowout',
        'Corrosion': 'Corrosion',
        'Tap Failure': 'Tap',
        'Crack': 'Crack',
        'Leak Outside Old Repair Band': 'Other',
        'Leak on Unused Fire Line': 'Other',
        'Crack - Main hit by Contractor': 'Crack',
    }
    df['Leak_Type_Clean'] = df['Leak_Type'].replace(leak_mapping)
    return df

# ====================== Apply the function ======================
df_clean_bozeman = clean_data(dfBozeman.copy())

# Save the cleaned dataset
df_clean_bozeman.to_csv(
    "../data/processed/Bozeman_watermain_breaks_cleaned_final.csv", 
    index=False
)

# Display first 10 rows
df_clean_bozeman.head(10)

,X,Y,Break_Date,Address,Pipe_Size_inch,Leak_Type,Pipe_ID,Detected,Install_Year,Break_Year,Age_at_Break,Age_Squared,Leak_Type_Clean
0,496675.559596,5.058910e+06,2013-01-28,431 N 4TH,6.0,Corrosion,334.0,No,1948,2013,65,4225,Corrosion
1,496631.156500,5.059796e+06,2012-08-11,1126 N 7TH (KMART),8.0,Longitudinal Break,204.0,No,1974,2012,38,1444,Longitudinal
4,497775.721701,5.058148e+06,2009-05-28,S WALLACE AVE& E OLIVE ST,6.0,Full Circle Break,408.0,No,1924,2009,85,7225,Circumferential
8,497263.634397,5.059113e+06,2011-05-18,Peach St. between Bozeman and Black,6.0,Leaking Joint,1179.0,No,1929,2011,82,6724,Joint
9,495063.790099,5.058329e+06,2008-12-04,1922 W BABCOCK,10.0,Longitudinal Break,1241.0,No,1971,2008,37,1369,Longitudinal
13,496878.991701,5.059050e+06,2014-07-23,Peach & Grand,6.0,Blow-out,1342.0,No,1903,2014,111,12321,Blowout
14,497729.421197,5.059782e+06,2011-05-10,PEAR ST BOSTER SOUTH OF,16.0,Leaking Joint,1662.0,No,1992,2011,19,361,Joint
15,497554.773501,5.060360e+06,2014-06-25,N Rouse near Bryant,4.0,Full Circle Break,2136.0,No,1950,2014,64,4096,Circumferential
16,498452.106398,5.058159e+06,2012-11-14,1207 E CURTISS,6.0,Full Circle Break,2329.0,No,1971,2012,41,1681,Circumferential
18,495457.321803,5.058784e+06,2008-12-19,300 BLOCK NORTH 16TH,6.0,Full Circle Break,2544.0,No,1966,2008,42,1764,Circumferential


### Statistics - Bozeman dataset after cleaning

In [16]:
# ====================== Bozeman Quick Validation ======================
print("=== Bozeman Feature Creation Summary ===")
print(df_clean_bozeman[['Break_Date', 'Install_Year', 'Break_Year', 
                        'Age_at_Break', 'Age_Squared', 
                        'Leak_Type','Leak_Type_Clean']].head(10).to_string())

print("\nAge_at_Break Statistics:")
print(df_clean_bozeman['Age_at_Break'].describe())

print("\nLeak_Type Distribution:")
print(df_clean_bozeman['Leak_Type'].value_counts())

# Check for negative ages - should be very few or zero
print(f"\nNumber of negative ages: {(df_clean_bozeman['Age_at_Break'] < 0).sum()}")

=== Bozeman Feature Creation Summary ===
   Break_Date  Install_Year  Break_Year  Age_at_Break  Age_Squared           Leak_Type  Leak_Type_Clean
0  2013-01-28          1948        2013            65         4225           Corrosion        Corrosion
1  2012-08-11          1974        2012            38         1444  Longitudinal Break     Longitudinal
4  2009-05-28          1924        2009            85         7225   Full Circle Break  Circumferential
8  2011-05-18          1929        2011            82         6724       Leaking Joint            Joint
9  2008-12-04          1971        2008            37         1369  Longitudinal Break     Longitudinal
13 2014-07-23          1903        2014           111        12321            Blow-out          Blowout
14 2011-05-10          1992        2011            19          361       Leaking Joint            Joint
15 2014-06-25          1950        2014            64         4096   Full Circle Break  Circumferential
16 2012-11-14          

### Statistics - Kitchener water main breaks after cleaning

In [18]:
# Display the cleaned datasets

print("\n=== Cleaned Bozeman Dataset ===")
print(df_clean_bozeman.head(10).to_string())


=== Cleaned Bozeman Dataset ===
                X             Y Break_Date                              Address  Pipe_Size_inch           Leak_Type  Pipe_ID Detected  Install_Year  Break_Year  Age_at_Break  Age_Squared  Leak_Type_Clean
0   496675.559596  5.058910e+06 2013-01-28                            431 N 4TH             6.0           Corrosion    334.0       No          1948        2013            65         4225        Corrosion
1   496631.156500  5.059796e+06 2012-08-11                   1126 N 7TH (KMART)             8.0  Longitudinal Break    204.0       No          1974        2012            38         1444     Longitudinal
4   497775.721701  5.058148e+06 2009-05-28            S WALLACE AVE& E OLIVE ST             6.0   Full Circle Break    408.0       No          1924        2009            85         7225  Circumferential
8   497263.634397  5.059113e+06 2011-05-18  Peach St. between Bozeman and Black             6.0       Leaking Joint   1179.0       No          1929    